# Integracion de Herramientas Externas y Protocolos de Comunicacion

**Objetivo:** Aprender a integrar herramientas externas con agentes de IA, comprender el Model Context Protocol (MCP) e implementar patrones de orquestacion de herramientas.

**Resultado de Aprendizaje:** RA2 - IL2.2: Sistemas de memoria e integracion de herramientas externas

---

## Configuracion del Entorno

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
if not os.getenv("GITHUB_TOKEN"):
    raise EnvironmentError("GITHUB_TOKEN no configurado. Copia .env.example a .env y agrega tu token.")
print("Entorno configurado correctamente")

Entorno configurado correctamente


In [2]:
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv("GITHUB_BASE_URL", "https://models.inference.ai.azure.com"),
    api_key=os.getenv("GITHUB_TOKEN")
)

MODELO = "gpt-4o-mini"
print(f"Cliente OpenAI configurado con modelo: {MODELO}")

Cliente OpenAI configurado con modelo: gpt-4o-mini


---

## Introduccion: Que son las Herramientas Externas para Agentes?

Los **agentes de IA** por si solos estan limitados a generar texto. Sin embargo, al integrarlos con **herramientas externas**, pueden:

- Realizar calculos precisos
- Consultar bases de datos y APIs
- Manipular archivos y sistemas
- Interactuar con servicios web
- Mantener estado entre interacciones

### Flujo Agente-Herramienta

```
┌──────────────┐     1. Solicitud      ┌──────────────┐
│              │ ───────────────────>   │              │
│   Usuario    │                        │    Agente    │
│              │ <───────────────────   │    (LLM)     │
└──────────────┘     5. Respuesta       └──────┬───────┘
                                               │
                                   2. Decide   │  4. Integra
                                      usar     │  resultado
                                   herramienta │
                                               │
                                        ┌──────▼───────┐
                                        │              │
                                        │ Herramientas │
                                        │  Externas    │
                                        │              │
                                        └──────────────┘
                                         3. Ejecuta y
                                            retorna
```

### Conceptos Clave

| Concepto | Descripcion |
|----------|-------------|
| **Function Calling** | Mecanismo que permite al LLM solicitar la ejecucion de funciones definidas |
| **Tool Definition** | Esquema JSON que describe una herramienta (nombre, descripcion, parametros) |
| **Tool Execution Loop** | Ciclo donde el agente decide usar herramientas, las ejecuta y procesa resultados |
| **MCP** | Model Context Protocol - protocolo estandar para conectar agentes con herramientas |

---

## Seccion 1: Definicion de Herramientas con Function Calling

El **Function Calling** permite definir herramientas que el modelo puede invocar. Cada herramienta se define con un **JSON Schema** que especifica:
- **Nombre** de la funcion
- **Descripcion** clara de lo que hace
- **Parametros** con tipos y validaciones

### 1.1 Definicion de herramientas con JSON Schema

In [3]:
import json

# Definicion de herramientas usando JSON Schema
herramientas = [
    {
        "type": "function",
        "function": {
            "name": "calculadora",
            "description": "Realiza operaciones matematicas basicas. Soporta suma, resta, multiplicacion, division y potencia.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operacion": {
                        "type": "string",
                        "enum": ["suma", "resta", "multiplicacion", "division", "potencia"],
                        "description": "La operacion matematica a realizar"
                    },
                    "a": {
                        "type": "number",
                        "description": "Primer operando"
                    },
                    "b": {
                        "type": "number",
                        "description": "Segundo operando"
                    }
                },
                "required": ["operacion", "a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "clima",
            "description": "Simula la consulta del clima actual para una ciudad dada. Retorna temperatura, condicion y humedad.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ciudad": {
                        "type": "string",
                        "description": "Nombre de la ciudad para consultar el clima"
                    }
                },
                "required": ["ciudad"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tomar_nota",
            "description": "Guarda una nota con un titulo y contenido. Util para recordar informacion importante.",
            "parameters": {
                "type": "object",
                "properties": {
                    "titulo": {
                        "type": "string",
                        "description": "Titulo breve de la nota"
                    },
                    "contenido": {
                        "type": "string",
                        "description": "Contenido detallado de la nota"
                    }
                },
                "required": ["titulo", "contenido"]
            }
        }
    }
]

print(f"Se definieron {len(herramientas)} herramientas:")
for h in herramientas:
    print(f"  - {h['function']['name']}: {h['function']['description'][:60]}...")

Se definieron 3 herramientas:
  - calculadora: Realiza operaciones matematicas basicas. Soporta suma, resta...
  - clima: Simula la consulta del clima actual para una ciudad dada. Re...
  - tomar_nota: Guarda una nota con un titulo y contenido. Util para recorda...


### 1.2 Implementacion de las funciones de herramientas

In [4]:
import random

# Almacen de notas (estado simple)
notas_guardadas = []


def ejecutar_calculadora(operacion: str, a: float, b: float) -> str:
    """Ejecuta una operacion matematica basica."""
    operaciones = {
        "suma": lambda x, y: x + y,
        "resta": lambda x, y: x - y,
        "multiplicacion": lambda x, y: x * y,
        "division": lambda x, y: x / y if y != 0 else "Error: division por cero",
        "potencia": lambda x, y: x ** y
    }
    if operacion not in operaciones:
        return json.dumps({"error": f"Operacion '{operacion}' no soportada"})
    resultado = operaciones[operacion](a, b)
    return json.dumps({
        "operacion": operacion,
        "a": a,
        "b": b,
        "resultado": resultado
    })


def ejecutar_clima(ciudad: str) -> str:
    """Simula una consulta de clima para una ciudad."""
    # Simulacion - en produccion esto seria una llamada a una API real
    condiciones = ["Soleado", "Nublado", "Lluvioso", "Parcialmente nublado", "Despejado"]
    datos_clima = {
        "ciudad": ciudad,
        "temperatura_celsius": round(random.uniform(5, 35), 1),
        "condicion": random.choice(condiciones),
        "humedad_porcentaje": random.randint(30, 90),
        "viento_kmh": round(random.uniform(0, 50), 1)
    }
    return json.dumps(datos_clima)


def ejecutar_tomar_nota(titulo: str, contenido: str) -> str:
    """Guarda una nota en el almacen."""
    from datetime import datetime
    nota = {
        "id": len(notas_guardadas) + 1,
        "titulo": titulo,
        "contenido": contenido,
        "fecha": datetime.now().isoformat()
    }
    notas_guardadas.append(nota)
    return json.dumps({"mensaje": f"Nota '{titulo}' guardada exitosamente", "id": nota["id"]})


# Mapeo de nombres de funcion a funciones ejecutables
FUNCIONES_DISPONIBLES = {
    "calculadora": ejecutar_calculadora,
    "clima": ejecutar_clima,
    "tomar_nota": ejecutar_tomar_nota
}

print("Funciones implementadas y registradas.")
# Prueba rapida
print("\nPrueba calculadora:", ejecutar_calculadora("suma", 5, 3))
print("Prueba clima:", ejecutar_clima("Santiago"))

Funciones implementadas y registradas.

Prueba calculadora: {"operacion": "suma", "a": 5, "b": 3, "resultado": 8}
Prueba clima: {"ciudad": "Santiago", "temperatura_celsius": 17.5, "condicion": "Parcialmente nublado", "humedad_porcentaje": 49, "viento_kmh": 8.7}


### 1.3 Loop de ejecucion de herramientas (Tool Execution Loop)

Este es el patron fundamental: el agente decide si necesita herramientas, las invoca y usa los resultados para generar su respuesta final.

In [5]:
def ejecutar_agente(mensaje_usuario: str, herramientas_def: list, funciones: dict, verbose: bool = True) -> str:
    """
    Ejecuta un agente con capacidad de usar herramientas.
    Implementa el loop completo: solicitud -> decision -> ejecucion -> respuesta.
    """
    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un asistente util que puede usar herramientas para ayudar al usuario. "
                "Cuando necesites realizar calculos, consultar el clima o tomar notas, "
                "usa las herramientas disponibles. Responde siempre en espanol."
            )
        },
        {"role": "user", "content": mensaje_usuario}
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"Usuario: {mensaje_usuario}")
        print(f"{'='*60}")

    # Loop de ejecucion - puede necesitar multiples iteraciones
    iteracion = 0
    max_iteraciones = 5  # Limite de seguridad

    while iteracion < max_iteraciones:
        iteracion += 1

        respuesta = client.chat.completions.create(
            model=MODELO,
            messages=mensajes,
            tools=herramientas_def,
            tool_choice="auto"
        )

        mensaje_respuesta = respuesta.choices[0].message

        # Si no hay llamadas a herramientas, el agente termino
        if not mensaje_respuesta.tool_calls:
            if verbose:
                print(f"\nRespuesta final del agente:")
                print(mensaje_respuesta.content)
            return mensaje_respuesta.content

        # Procesar cada llamada a herramienta
        mensajes.append(mensaje_respuesta)

        for tool_call in mensaje_respuesta.tool_calls:
            nombre_funcion = tool_call.function.name
            argumentos = json.loads(tool_call.function.arguments)

            if verbose:
                print(f"\n[Iteracion {iteracion}] Herramienta invocada: {nombre_funcion}")
                print(f"  Argumentos: {argumentos}")

            # Ejecutar la funcion
            if nombre_funcion in funciones:
                resultado = funciones[nombre_funcion](**argumentos)
            else:
                resultado = json.dumps({"error": f"Funcion '{nombre_funcion}' no encontrada"})

            if verbose:
                print(f"  Resultado: {resultado}")

            # Agregar resultado al historial
            mensajes.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": resultado
            })

    return "Se alcanzo el limite maximo de iteraciones."


print("Funcion ejecutar_agente definida correctamente.")

Funcion ejecutar_agente definida correctamente.


In [6]:
# Ejemplo 1: Uso de la calculadora
ejecutar_agente(
    "Cuanto es 125 multiplicado por 48, y luego sumale 320?",
    herramientas,
    FUNCIONES_DISPONIBLES
)


Usuario: Cuanto es 125 multiplicado por 48, y luego sumale 320?

[Iteracion 1] Herramienta invocada: calculadora
  Argumentos: {'operacion': 'multiplicacion', 'a': 125, 'b': 48}
  Resultado: {"operacion": "multiplicacion", "a": 125, "b": 48, "resultado": 6000}

[Iteracion 2] Herramienta invocada: calculadora
  Argumentos: {'operacion': 'suma', 'a': 6000, 'b': 320}
  Resultado: {"operacion": "suma", "a": 6000, "b": 320, "resultado": 6320}

Respuesta final del agente:
El resultado de 125 multiplicado por 48 es 6000, y al sumarle 320 da un total de 6320.


'El resultado de 125 multiplicado por 48 es 6000, y al sumarle 320 da un total de 6320.'

In [7]:
# Ejemplo 2: Consulta de clima
ejecutar_agente(
    "Como esta el clima en Santiago y en Valparaiso?",
    herramientas,
    FUNCIONES_DISPONIBLES
)


Usuario: Como esta el clima en Santiago y en Valparaiso?

[Iteracion 1] Herramienta invocada: clima
  Argumentos: {'ciudad': 'Santiago'}
  Resultado: {"ciudad": "Santiago", "temperatura_celsius": 9.7, "condicion": "Lluvioso", "humedad_porcentaje": 86, "viento_kmh": 3.2}

[Iteracion 1] Herramienta invocada: clima
  Argumentos: {'ciudad': 'Valparaiso'}
  Resultado: {"ciudad": "Valparaiso", "temperatura_celsius": 34.6, "condicion": "Soleado", "humedad_porcentaje": 75, "viento_kmh": 25.4}

Respuesta final del agente:
El clima en las ciudades que consultaste es el siguiente:

### Santiago:
- **Temperatura:** 9.7 °C
- **Condición:** Lluvioso
- **Humedad:** 86%
- **Viento:** 3.2 km/h

### Valparaíso:
- **Temperatura:** 34.6 °C
- **Condición:** Soleado
- **Humedad:** 75%
- **Viento:** 25.4 km/h

Si necesitas más información, no dudes en preguntar.


'El clima en las ciudades que consultaste es el siguiente:\n\n### Santiago:\n- **Temperatura:** 9.7 °C\n- **Condición:** Lluvioso\n- **Humedad:** 86%\n- **Viento:** 3.2 km/h\n\n### Valparaíso:\n- **Temperatura:** 34.6 °C\n- **Condición:** Soleado\n- **Humedad:** 75%\n- **Viento:** 25.4 km/h\n\nSi necesitas más información, no dudes en preguntar.'

In [8]:
# Ejemplo 3: Combinacion de herramientas
ejecutar_agente(
    "Consulta el clima en Concepcion y toma nota del resultado con el titulo 'Clima Concepcion'",
    herramientas,
    FUNCIONES_DISPONIBLES
)


Usuario: Consulta el clima en Concepcion y toma nota del resultado con el titulo 'Clima Concepcion'

[Iteracion 1] Herramienta invocada: clima
  Argumentos: {'ciudad': 'Concepcion'}
  Resultado: {"ciudad": "Concepcion", "temperatura_celsius": 11.1, "condicion": "Lluvioso", "humedad_porcentaje": 62, "viento_kmh": 49.2}

[Iteracion 2] Herramienta invocada: tomar_nota
  Argumentos: {'titulo': 'Clima Concepcion', 'contenido': 'Temperatura: 11.1 °C\nCondición: Lluvioso\nHumedad: 62%\nViento: 49.2 km/h'}
  Resultado: {"mensaje": "Nota 'Clima Concepcion' guardada exitosamente", "id": 1}

Respuesta final del agente:
He consultado el clima en Concepción. Aquí tienes los resultados:

- **Temperatura**: 11.1 °C
- **Condición**: Lluvioso
- **Humedad**: 62%
- **Viento**: 49.2 km/h

También he guardado la nota con el título "Clima Concepcion" exitosamente.


'He consultado el clima en Concepción. Aquí tienes los resultados:\n\n- **Temperatura**: 11.1 °C\n- **Condición**: Lluvioso\n- **Humedad**: 62%\n- **Viento**: 49.2 km/h\n\nTambién he guardado la nota con el título "Clima Concepcion" exitosamente.'

---

## Seccion 2: Registro Dinamico de Herramientas

En aplicaciones reales, las herramientas disponibles pueden cambiar en tiempo de ejecucion. Un **ToolRegistry** permite:
- Agregar herramientas dinamicamente
- Eliminar herramientas cuando ya no se necesitan
- Validar que las herramientas esten correctamente definidas
- Listar herramientas disponibles para el agente

In [9]:
from typing import Callable, Any


class RegistroHerramientas:
    """
    Registro dinamico de herramientas para agentes.
    Permite agregar, remover y descubrir herramientas en tiempo de ejecucion.
    """

    def __init__(self):
        self._herramientas: dict[str, dict] = {}  # nombre -> definicion JSON Schema
        self._funciones: dict[str, Callable] = {}   # nombre -> funcion ejecutable

    def registrar(self, nombre: str, descripcion: str, parametros: dict, funcion: Callable) -> None:
        """Registra una nueva herramienta en el sistema."""
        # Validar parametros requeridos
        if not nombre or not descripcion:
            raise ValueError("El nombre y la descripcion son obligatorios")
        if not callable(funcion):
            raise ValueError("La funcion debe ser callable")

        self._herramientas[nombre] = {
            "type": "function",
            "function": {
                "name": nombre,
                "description": descripcion,
                "parameters": parametros
            }
        }
        self._funciones[nombre] = funcion
        print(f"  [+] Herramienta '{nombre}' registrada exitosamente")

    def desregistrar(self, nombre: str) -> bool:
        """Elimina una herramienta del registro."""
        if nombre in self._herramientas:
            del self._herramientas[nombre]
            del self._funciones[nombre]
            print(f"  [-] Herramienta '{nombre}' eliminada")
            return True
        print(f"  [!] Herramienta '{nombre}' no encontrada")
        return False

    def listar(self) -> list[str]:
        """Lista los nombres de herramientas disponibles."""
        return list(self._herramientas.keys())

    def obtener_definiciones(self) -> list[dict]:
        """Retorna las definiciones JSON Schema para el API."""
        return list(self._herramientas.values())

    def obtener_funciones(self) -> dict[str, Callable]:
        """Retorna el mapeo nombre -> funcion."""
        return self._funciones.copy()

    def ejecutar(self, nombre: str, **kwargs) -> str:
        """Ejecuta una herramienta por nombre con los argumentos dados."""
        if nombre not in self._funciones:
            return json.dumps({"error": f"Herramienta '{nombre}' no registrada"})
        return self._funciones[nombre](**kwargs)

    def info(self, nombre: str) -> dict | None:
        """Retorna la informacion de una herramienta especifica."""
        if nombre in self._herramientas:
            return self._herramientas[nombre]["function"]
        return None

    def __len__(self):
        return len(self._herramientas)

    def __repr__(self):
        return f"RegistroHerramientas({len(self)} herramientas: {self.listar()})"


print("Clase RegistroHerramientas definida.")

Clase RegistroHerramientas definida.


In [10]:
# Crear registro y agregar herramientas
registro = RegistroHerramientas()

print("Registrando herramientas...\n")

# Registrar calculadora
registro.registrar(
    nombre="calculadora",
    descripcion="Realiza operaciones matematicas: suma, resta, multiplicacion, division, potencia.",
    parametros={
        "type": "object",
        "properties": {
            "operacion": {"type": "string", "enum": ["suma", "resta", "multiplicacion", "division", "potencia"]},
            "a": {"type": "number", "description": "Primer operando"},
            "b": {"type": "number", "description": "Segundo operando"}
        },
        "required": ["operacion", "a", "b"]
    },
    funcion=ejecutar_calculadora
)

# Registrar clima
registro.registrar(
    nombre="clima",
    descripcion="Consulta el clima actual de una ciudad.",
    parametros={
        "type": "object",
        "properties": {
            "ciudad": {"type": "string", "description": "Nombre de la ciudad"}
        },
        "required": ["ciudad"]
    },
    funcion=ejecutar_clima
)

# Registrar una herramienta nueva: conversor de unidades
def ejecutar_conversor(valor: float, de_unidad: str, a_unidad: str) -> str:
    """Convierte entre unidades de temperatura."""
    conversiones = {
        ("celsius", "fahrenheit"): lambda v: v * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5/9,
        ("celsius", "kelvin"): lambda v: v + 273.15,
        ("kelvin", "celsius"): lambda v: v - 273.15,
    }
    clave = (de_unidad.lower(), a_unidad.lower())
    if clave in conversiones:
        resultado = round(conversiones[clave](valor), 2)
        return json.dumps({"valor_original": valor, "de": de_unidad, "a": a_unidad, "resultado": resultado})
    return json.dumps({"error": f"Conversion de {de_unidad} a {a_unidad} no soportada"})

registro.registrar(
    nombre="conversor_temperatura",
    descripcion="Convierte temperaturas entre Celsius, Fahrenheit y Kelvin.",
    parametros={
        "type": "object",
        "properties": {
            "valor": {"type": "number", "description": "Valor numerico a convertir"},
            "de_unidad": {"type": "string", "enum": ["celsius", "fahrenheit", "kelvin"]},
            "a_unidad": {"type": "string", "enum": ["celsius", "fahrenheit", "kelvin"]}
        },
        "required": ["valor", "de_unidad", "a_unidad"]
    },
    funcion=ejecutar_conversor
)

print(f"\nRegistro actual: {registro}")
print(f"Herramientas disponibles: {registro.listar()}")

Registrando herramientas...

  [+] Herramienta 'calculadora' registrada exitosamente
  [+] Herramienta 'clima' registrada exitosamente
  [+] Herramienta 'conversor_temperatura' registrada exitosamente

Registro actual: RegistroHerramientas(3 herramientas: ['calculadora', 'clima', 'conversor_temperatura'])
Herramientas disponibles: ['calculadora', 'clima', 'conversor_temperatura']


In [11]:
# Usar el registro con el agente
ejecutar_agente(
    "Consulta el clima en Santiago y convierte la temperatura a Fahrenheit",
    registro.obtener_definiciones(),
    registro.obtener_funciones()
)


Usuario: Consulta el clima en Santiago y convierte la temperatura a Fahrenheit

[Iteracion 1] Herramienta invocada: clima
  Argumentos: {'ciudad': 'Santiago'}
  Resultado: {"ciudad": "Santiago", "temperatura_celsius": 10.8, "condicion": "Lluvioso", "humedad_porcentaje": 85, "viento_kmh": 49.4}

[Iteracion 2] Herramienta invocada: conversor_temperatura
  Argumentos: {'valor': 10.8, 'de_unidad': 'celsius', 'a_unidad': 'fahrenheit'}
  Resultado: {"valor_original": 10.8, "de": "celsius", "a": "fahrenheit", "resultado": 51.44}

Respuesta final del agente:
El clima en Santiago es lluvioso, con una temperatura de 10.8 °C. Al convertir esta temperatura a Fahrenheit, obtenemos 51.44 °F. La humedad es del 85% y la velocidad del viento es de 49.4 km/h.


'El clima en Santiago es lluvioso, con una temperatura de 10.8 °C. Al convertir esta temperatura a Fahrenheit, obtenemos 51.44 °F. La humedad es del 85% y la velocidad del viento es de 49.4 km/h.'

In [12]:
# Demostrar eliminacion dinamica
print("Antes de eliminar:", registro.listar())
registro.desregistrar("conversor_temperatura")
print("Despues de eliminar:", registro.listar())

# El agente ya no puede usar el conversor
ejecutar_agente(
    "Convierte 100 grados celsius a fahrenheit",
    registro.obtener_definiciones(),
    registro.obtener_funciones()
)

Antes de eliminar: ['calculadora', 'clima', 'conversor_temperatura']
  [-] Herramienta 'conversor_temperatura' eliminada
Despues de eliminar: ['calculadora', 'clima']

Usuario: Convierte 100 grados celsius a fahrenheit

[Iteracion 1] Herramienta invocada: calculadora
  Argumentos: {'operacion': 'multiplicacion', 'a': 100, 'b': 9}
  Resultado: {"operacion": "multiplicacion", "a": 100, "b": 9, "resultado": 900}

[Iteracion 2] Herramienta invocada: calculadora
  Argumentos: {'operacion': 'division', 'a': 900, 'b': 5}
  Resultado: {"operacion": "division", "a": 900, "b": 5, "resultado": 180.0}

[Iteracion 3] Herramienta invocada: calculadora
  Argumentos: {'operacion': 'suma', 'a': 180, 'b': 32}
  Resultado: {"operacion": "suma", "a": 180, "b": 32, "resultado": 212}

Respuesta final del agente:
100 grados Celsius son equivalentes a 212 grados Fahrenheit.


'100 grados Celsius son equivalentes a 212 grados Fahrenheit.'

---

## Seccion 3: Herramientas con Estado (Stateful Tools)

Algunas herramientas necesitan **mantener estado** entre llamadas. Por ejemplo:
- Un **carrito de compras** que acumula productos
- Una **lista de tareas** que persiste entre interacciones
- Un **historial de busquedas** que recuerda consultas anteriores

Implementaremos herramientas con estado usando clases de Python.

In [13]:
from datetime import datetime


class CarritoCompras:
    """Herramienta con estado: carrito de compras."""

    def __init__(self):
        self.items: list[dict] = []

    def agregar(self, producto: str, cantidad: int, precio_unitario: float) -> str:
        """Agrega un producto al carrito."""
        item = {
            "producto": producto,
            "cantidad": cantidad,
            "precio_unitario": precio_unitario,
            "subtotal": cantidad * precio_unitario
        }
        self.items.append(item)
        return json.dumps({
            "accion": "agregado",
            "producto": producto,
            "cantidad": cantidad,
            "subtotal": item["subtotal"],
            "items_en_carrito": len(self.items)
        })

    def ver(self) -> str:
        """Muestra el contenido del carrito."""
        if not self.items:
            return json.dumps({"mensaje": "El carrito esta vacio", "items": [], "total": 0})
        total = sum(item["subtotal"] for item in self.items)
        return json.dumps({
            "items": self.items,
            "cantidad_items": len(self.items),
            "total": round(total, 2)
        })

    def eliminar(self, producto: str) -> str:
        """Elimina un producto del carrito."""
        for i, item in enumerate(self.items):
            if item["producto"].lower() == producto.lower():
                eliminado = self.items.pop(i)
                return json.dumps({"accion": "eliminado", "producto": eliminado["producto"]})
        return json.dumps({"error": f"Producto '{producto}' no encontrado en el carrito"})


class ListaTareas:
    """Herramienta con estado: lista de tareas."""

    def __init__(self):
        self.tareas: list[dict] = []
        self._id_counter = 0

    def agregar(self, descripcion: str, prioridad: str = "media") -> str:
        """Agrega una tarea a la lista."""
        self._id_counter += 1
        tarea = {
            "id": self._id_counter,
            "descripcion": descripcion,
            "prioridad": prioridad,
            "completada": False,
            "fecha_creacion": datetime.now().isoformat()
        }
        self.tareas.append(tarea)
        return json.dumps({"accion": "tarea_creada", "id": tarea["id"], "descripcion": descripcion})

    def completar(self, id_tarea: int) -> str:
        """Marca una tarea como completada."""
        for tarea in self.tareas:
            if tarea["id"] == id_tarea:
                tarea["completada"] = True
                tarea["fecha_completada"] = datetime.now().isoformat()
                return json.dumps({"accion": "tarea_completada", "id": id_tarea, "descripcion": tarea["descripcion"]})
        return json.dumps({"error": f"Tarea con id {id_tarea} no encontrada"})

    def listar(self, solo_pendientes: bool = False) -> str:
        """Lista todas las tareas o solo las pendientes."""
        tareas = self.tareas
        if solo_pendientes:
            tareas = [t for t in tareas if not t["completada"]]
        return json.dumps({
            "tareas": tareas,
            "total": len(self.tareas),
            "pendientes": sum(1 for t in self.tareas if not t["completada"]),
            "completadas": sum(1 for t in self.tareas if t["completada"])
        })


print("Clases CarritoCompras y ListaTareas definidas.")

Clases CarritoCompras y ListaTareas definidas.


In [14]:
# Crear instancias con estado
carrito = CarritoCompras()
tareas = ListaTareas()

# Registrar herramientas con estado en un nuevo registro
registro_stateful = RegistroHerramientas()

print("Registrando herramientas con estado...\n")

# Herramientas del carrito
registro_stateful.registrar(
    nombre="carrito_agregar",
    descripcion="Agrega un producto al carrito de compras con cantidad y precio unitario.",
    parametros={
        "type": "object",
        "properties": {
            "producto": {"type": "string", "description": "Nombre del producto"},
            "cantidad": {"type": "integer", "description": "Cantidad a agregar"},
            "precio_unitario": {"type": "number", "description": "Precio por unidad"}
        },
        "required": ["producto", "cantidad", "precio_unitario"]
    },
    funcion=carrito.agregar
)

registro_stateful.registrar(
    nombre="carrito_ver",
    descripcion="Muestra el contenido actual del carrito de compras con el total.",
    parametros={"type": "object", "properties": {}},
    funcion=carrito.ver
)

registro_stateful.registrar(
    nombre="carrito_eliminar",
    descripcion="Elimina un producto del carrito de compras por su nombre.",
    parametros={
        "type": "object",
        "properties": {
            "producto": {"type": "string", "description": "Nombre del producto a eliminar"}
        },
        "required": ["producto"]
    },
    funcion=carrito.eliminar
)

# Herramientas de tareas
registro_stateful.registrar(
    nombre="tarea_agregar",
    descripcion="Agrega una nueva tarea a la lista con descripcion y prioridad (alta, media, baja).",
    parametros={
        "type": "object",
        "properties": {
            "descripcion": {"type": "string", "description": "Descripcion de la tarea"},
            "prioridad": {"type": "string", "enum": ["alta", "media", "baja"], "description": "Nivel de prioridad"}
        },
        "required": ["descripcion"]
    },
    funcion=tareas.agregar
)

registro_stateful.registrar(
    nombre="tarea_completar",
    descripcion="Marca una tarea como completada usando su ID numerico.",
    parametros={
        "type": "object",
        "properties": {
            "id_tarea": {"type": "integer", "description": "ID de la tarea a completar"}
        },
        "required": ["id_tarea"]
    },
    funcion=tareas.completar
)

registro_stateful.registrar(
    nombre="tarea_listar",
    descripcion="Lista todas las tareas. Puede filtrar solo las pendientes.",
    parametros={
        "type": "object",
        "properties": {
            "solo_pendientes": {"type": "boolean", "description": "Si es true, muestra solo tareas pendientes"}
        }
    },
    funcion=tareas.listar
)

print(f"\nRegistro: {registro_stateful}")

Registrando herramientas con estado...

  [+] Herramienta 'carrito_agregar' registrada exitosamente
  [+] Herramienta 'carrito_ver' registrada exitosamente
  [+] Herramienta 'carrito_eliminar' registrada exitosamente
  [+] Herramienta 'tarea_agregar' registrada exitosamente
  [+] Herramienta 'tarea_completar' registrada exitosamente
  [+] Herramienta 'tarea_listar' registrada exitosamente

Registro: RegistroHerramientas(6 herramientas: ['carrito_agregar', 'carrito_ver', 'carrito_eliminar', 'tarea_agregar', 'tarea_completar', 'tarea_listar'])


In [15]:
# Interaccion 1: Agregar productos al carrito
ejecutar_agente(
    "Agrega al carrito: 3 manzanas a $500 cada una y 2 litros de leche a $1200 cada uno",
    registro_stateful.obtener_definiciones(),
    registro_stateful.obtener_funciones()
)


Usuario: Agrega al carrito: 3 manzanas a $500 cada una y 2 litros de leche a $1200 cada uno

[Iteracion 1] Herramienta invocada: carrito_agregar
  Argumentos: {'producto': 'manzanas', 'cantidad': 3, 'precio_unitario': 500}
  Resultado: {"accion": "agregado", "producto": "manzanas", "cantidad": 3, "subtotal": 1500, "items_en_carrito": 1}

[Iteracion 1] Herramienta invocada: carrito_agregar
  Argumentos: {'producto': 'leche', 'cantidad': 2, 'precio_unitario': 1200}
  Resultado: {"accion": "agregado", "producto": "leche", "cantidad": 2, "subtotal": 2400, "items_en_carrito": 2}

Respuesta final del agente:
He agregado al carrito:

- 3 manzanas a $500 cada una (subtotal: $1500)
- 2 litros de leche a $1200 cada uno (subtotal: $2400)

Si deseas ver el total o continuar con otra acción, házmelo saber.


'He agregado al carrito:\n\n- 3 manzanas a $500 cada una (subtotal: $1500)\n- 2 litros de leche a $1200 cada uno (subtotal: $2400)\n\nSi deseas ver el total o continuar con otra acción, házmelo saber.'

In [16]:
# Interaccion 2: Ver el carrito (el estado persiste!)
ejecutar_agente(
    "Muestrame que hay en el carrito de compras",
    registro_stateful.obtener_definiciones(),
    registro_stateful.obtener_funciones()
)


Usuario: Muestrame que hay en el carrito de compras

[Iteracion 1] Herramienta invocada: carrito_ver
  Argumentos: {}
  Resultado: {"items": [{"producto": "manzanas", "cantidad": 3, "precio_unitario": 500, "subtotal": 1500}, {"producto": "leche", "cantidad": 2, "precio_unitario": 1200, "subtotal": 2400}], "cantidad_items": 2, "total": 3900}

Respuesta final del agente:
En el carrito de compras hay los siguientes productos:

1. **Manzanas**
   - Cantidad: 3
   - Precio por unidad: 500
   - Subtotal: 1500

2. **Leche**
   - Cantidad: 2
   - Precio por unidad: 1200
   - Subtotal: 2400

### Resumen:
- Total de artículos en el carrito: 2
- Total a pagar: 3900


'En el carrito de compras hay los siguientes productos:\n\n1. **Manzanas**\n   - Cantidad: 3\n   - Precio por unidad: 500\n   - Subtotal: 1500\n\n2. **Leche**\n   - Cantidad: 2\n   - Precio por unidad: 1200\n   - Subtotal: 2400\n\n### Resumen:\n- Total de artículos en el carrito: 2\n- Total a pagar: 3900'

In [17]:
# Interaccion 3: Usar tareas
ejecutar_agente(
    "Crea tres tareas: 'Comprar ingredientes' con prioridad alta, 'Preparar presentacion' con prioridad media, y 'Limpiar escritorio' con prioridad baja. Luego muestra la lista.",
    registro_stateful.obtener_definiciones(),
    registro_stateful.obtener_funciones()
)


Usuario: Crea tres tareas: 'Comprar ingredientes' con prioridad alta, 'Preparar presentacion' con prioridad media, y 'Limpiar escritorio' con prioridad baja. Luego muestra la lista.

[Iteracion 1] Herramienta invocada: tarea_agregar
  Argumentos: {'descripcion': 'Comprar ingredientes', 'prioridad': 'alta'}
  Resultado: {"accion": "tarea_creada", "id": 1, "descripcion": "Comprar ingredientes"}

[Iteracion 1] Herramienta invocada: tarea_agregar
  Argumentos: {'descripcion': 'Preparar presentacion', 'prioridad': 'media'}
  Resultado: {"accion": "tarea_creada", "id": 2, "descripcion": "Preparar presentacion"}

[Iteracion 1] Herramienta invocada: tarea_agregar
  Argumentos: {'descripcion': 'Limpiar escritorio', 'prioridad': 'baja'}
  Resultado: {"accion": "tarea_creada", "id": 3, "descripcion": "Limpiar escritorio"}

[Iteracion 2] Herramienta invocada: tarea_listar
  Argumentos: {}
  Resultado: {"tareas": [{"id": 1, "descripcion": "Comprar ingredientes", "prioridad": "alta", "completada": 

'He creado las siguientes tareas:\n\n1. **Comprar ingredientes** - Prioridad: Alta\n2. **Preparar presentación** - Prioridad: Media\n3. **Limpiar escritorio** - Prioridad: Baja\n\nAquí están las tareas en la lista:\n\n- **ID 1**: Comprar ingredientes (Prioridad: Alta) - Pendiente\n- **ID 2**: Preparar presentación (Prioridad: Media) - Pendiente\n- **ID 3**: Limpiar escritorio (Prioridad: Baja) - Pendiente\n\nTotal de tareas: 3 (Todas pendientes)'

In [18]:
# Interaccion 4: Completar tarea y verificar estado
ejecutar_agente(
    "Marca la tarea 1 como completada y luego muestrame solo las tareas pendientes",
    registro_stateful.obtener_definiciones(),
    registro_stateful.obtener_funciones()
)


Usuario: Marca la tarea 1 como completada y luego muestrame solo las tareas pendientes

[Iteracion 1] Herramienta invocada: tarea_completar
  Argumentos: {'id_tarea': 1}
  Resultado: {"accion": "tarea_completada", "id": 1, "descripcion": "Comprar ingredientes"}

[Iteracion 2] Herramienta invocada: tarea_listar
  Argumentos: {'solo_pendientes': True}
  Resultado: {"tareas": [{"id": 2, "descripcion": "Preparar presentacion", "prioridad": "media", "completada": false, "fecha_creacion": "2026-04-28T12:44:26.253787"}, {"id": 3, "descripcion": "Limpiar escritorio", "prioridad": "baja", "completada": false, "fecha_creacion": "2026-04-28T12:44:26.253825"}], "total": 3, "pendientes": 2, "completadas": 1}

Respuesta final del agente:
He marcado la tarea "Comprar ingredientes" como completada. Actualmente, las tareas pendientes son:

1. **Preparar presentación** (Prioridad: media)
2. **Limpiar escritorio** (Prioridad: baja) 

Si necesitas algo más, házmelo saber.


'He marcado la tarea "Comprar ingredientes" como completada. Actualmente, las tareas pendientes son:\n\n1. **Preparar presentación** (Prioridad: media)\n2. **Limpiar escritorio** (Prioridad: baja) \n\nSi necesitas algo más, házmelo saber.'

---

## Seccion 4: Orquestacion de Multiples Herramientas

La **orquestacion** implica coordinar multiples herramientas para completar tareas complejas. El agente debe:
1. **Descomponer** la tarea en pasos
2. **Seleccionar** la herramienta correcta para cada paso
3. **Encadenar** los resultados de un paso al siguiente
4. **Resolver dependencias** entre herramientas

In [19]:
# Crear un registro con todas las herramientas para orquestacion
registro_completo = RegistroHerramientas()

print("Registrando conjunto completo de herramientas para orquestacion...\n")

# Calculadora
registro_completo.registrar(
    "calculadora",
    "Realiza operaciones matematicas: suma, resta, multiplicacion, division, potencia.",
    {
        "type": "object",
        "properties": {
            "operacion": {"type": "string", "enum": ["suma", "resta", "multiplicacion", "division", "potencia"]},
            "a": {"type": "number"},
            "b": {"type": "number"}
        },
        "required": ["operacion", "a", "b"]
    },
    ejecutar_calculadora
)

# Clima
registro_completo.registrar(
    "clima",
    "Consulta el clima actual de una ciudad. Retorna temperatura, condicion, humedad y viento.",
    {
        "type": "object",
        "properties": {"ciudad": {"type": "string"}},
        "required": ["ciudad"]
    },
    ejecutar_clima
)

# Conversor de moneda (nueva herramienta)
def ejecutar_conversor_moneda(monto: float, de_moneda: str, a_moneda: str) -> str:
    """Simula conversion de moneda."""
    tasas = {
        ("CLP", "USD"): 0.0011,
        ("USD", "CLP"): 920.0,
        ("CLP", "EUR"): 0.0010,
        ("EUR", "CLP"): 1000.0,
        ("USD", "EUR"): 0.92,
        ("EUR", "USD"): 1.09,
    }
    clave = (de_moneda.upper(), a_moneda.upper())
    if clave in tasas:
        resultado = round(monto * tasas[clave], 2)
        return json.dumps({"monto_original": monto, "de": de_moneda, "a": a_moneda, "resultado": resultado, "tasa": tasas[clave]})
    return json.dumps({"error": f"Conversion {de_moneda} -> {a_moneda} no disponible"})

registro_completo.registrar(
    "conversor_moneda",
    "Convierte montos entre monedas: CLP (peso chileno), USD (dolar), EUR (euro).",
    {
        "type": "object",
        "properties": {
            "monto": {"type": "number", "description": "Monto a convertir"},
            "de_moneda": {"type": "string", "enum": ["CLP", "USD", "EUR"]},
            "a_moneda": {"type": "string", "enum": ["CLP", "USD", "EUR"]}
        },
        "required": ["monto", "de_moneda", "a_moneda"]
    },
    ejecutar_conversor_moneda
)

# Notas
registro_completo.registrar(
    "tomar_nota",
    "Guarda una nota con titulo y contenido para referencia futura.",
    {
        "type": "object",
        "properties": {
            "titulo": {"type": "string"},
            "contenido": {"type": "string"}
        },
        "required": ["titulo", "contenido"]
    },
    ejecutar_tomar_nota
)

print(f"\nRegistro completo: {registro_completo}")

Registrando conjunto completo de herramientas para orquestacion...

  [+] Herramienta 'calculadora' registrada exitosamente
  [+] Herramienta 'clima' registrada exitosamente
  [+] Herramienta 'conversor_moneda' registrada exitosamente
  [+] Herramienta 'tomar_nota' registrada exitosamente

Registro completo: RegistroHerramientas(4 herramientas: ['calculadora', 'clima', 'conversor_moneda', 'tomar_nota'])


In [20]:
# Tarea multi-paso: el agente debe encadenar herramientas
ejecutar_agente(
    (
        "Necesito planificar un viaje. Primero consulta el clima en Santiago y en Buenos Aires. "
        "Luego calcula cuanto cuesta un hotel de 5 noches a $45.000 CLP por noche, "
        "convierte ese total a dolares, y toma una nota con el resumen del plan de viaje "
        "incluyendo el clima y el costo."
    ),
    registro_completo.obtener_definiciones(),
    registro_completo.obtener_funciones()
)


Usuario: Necesito planificar un viaje. Primero consulta el clima en Santiago y en Buenos Aires. Luego calcula cuanto cuesta un hotel de 5 noches a $45.000 CLP por noche, convierte ese total a dolares, y toma una nota con el resumen del plan de viaje incluyendo el clima y el costo.

[Iteracion 1] Herramienta invocada: clima
  Argumentos: {'ciudad': 'Santiago'}
  Resultado: {"ciudad": "Santiago", "temperatura_celsius": 26.7, "condicion": "Parcialmente nublado", "humedad_porcentaje": 78, "viento_kmh": 8.4}

[Iteracion 1] Herramienta invocada: clima
  Argumentos: {'ciudad': 'Buenos Aires'}
  Resultado: {"ciudad": "Buenos Aires", "temperatura_celsius": 22.4, "condicion": "Nublado", "humedad_porcentaje": 51, "viento_kmh": 34.5}

[Iteracion 1] Herramienta invocada: calculadora
  Argumentos: {'operacion': 'multiplicacion', 'a': 45000, 'b': 5}
  Resultado: {"operacion": "multiplicacion", "a": 45000, "b": 5, "resultado": 225000}

[Iteracion 2] Herramienta invocada: conversor_moneda
  Argumentos

'He recopilado la información para tu viaje:\n\n### Clima:\n- **Santiago:** 26.7°C, Parcialmente nublado, Humedad: 78%, Viento: 8.4 km/h.\n- **Buenos Aires:** 22.4°C, Nublado, Humedad: 51%, Viento: 34.5 km/h.\n\n### Costo del hotel:\n- Precio por noche: $45.000 CLP.\n- Total por 5 noches: $225.000 CLP (aproximadamente $247.5 USD).\n\nAdemás, tomé una nota con el resumen del plan de viaje, que ha sido guardada exitosamente. Si necesitas algo más, no dudes en decírmelo.'

In [21]:
# Otra tarea multi-paso con dependencias
ejecutar_agente(
    (
        "Quiero saber cuanto es 15% de propina sobre una cuenta de $87.500 CLP. "
        "Luego convierte tanto la cuenta como la propina a euros. "
        "Finalmente, toma nota del calculo completo."
    ),
    registro_completo.obtener_definiciones(),
    registro_completo.obtener_funciones()
)


Usuario: Quiero saber cuanto es 15% de propina sobre una cuenta de $87.500 CLP. Luego convierte tanto la cuenta como la propina a euros. Finalmente, toma nota del calculo completo.

[Iteracion 1] Herramienta invocada: calculadora
  Argumentos: {'operacion': 'multiplicacion', 'a': 87500, 'b': 0.15}
  Resultado: {"operacion": "multiplicacion", "a": 87500, "b": 0.15, "resultado": 13125.0}

[Iteracion 2] Herramienta invocada: conversor_moneda
  Argumentos: {'monto': 87500, 'de_moneda': 'CLP', 'a_moneda': 'EUR'}
  Resultado: {"monto_original": 87500, "de": "CLP", "a": "EUR", "resultado": 87.5, "tasa": 0.001}

[Iteracion 2] Herramienta invocada: conversor_moneda
  Argumentos: {'monto': 13125, 'de_moneda': 'CLP', 'a_moneda': 'EUR'}
  Resultado: {"monto_original": 13125, "de": "CLP", "a": "EUR", "resultado": 13.12, "tasa": 0.001}

[Iteracion 3] Herramienta invocada: tomar_nota
  Argumentos: {'titulo': 'Cálculo de propina', 'contenido': 'Cuenta original: $87.500 CLP\nPropina (15%): $13.125 CLP

'El cálculo de la propina sobre una cuenta de $87.500 CLP es el siguiente:\n\n1. **Propina (15%)**: $13.125 CLP\n2. **Conversión a euros**:\n   - Cuenta: €87.5 EUR\n   - Propina: €13.12 EUR\n\nHe guardado la nota con el cálculo completo. Si necesitas algo más, ¡déjamelo saber!'

---

## Seccion 5: Introduccion a MCP (Model Context Protocol)

### Que es MCP?

El **Model Context Protocol (MCP)** es un protocolo abierto estandarizado para conectar agentes de IA con fuentes de datos y herramientas externas. Fue propuesto por Anthropic y busca resolver el problema de la **integracion fragmentada** de herramientas.

### Arquitectura MCP

```
┌─────────────────────────────────────────────────────────────┐
│                    Aplicacion Host                          │
│  (IDE, Chatbot, Asistente)                                 │
│                                                             │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐        │
│  │ MCP Cliente  │  │ MCP Cliente  │  │ MCP Cliente  │       │
│  │     A        │  │     B        │  │     C        │       │
│  └──────┬───────┘  └──────┬───────┘  └──────┬───────┘      │
└─────────┼─────────────────┼─────────────────┼───────────────┘
          │                 │                 │
          │   Protocolo     │   Protocolo     │   Protocolo
          │   MCP           │   MCP           │   MCP
          │                 │                 │
┌─────────▼──────┐ ┌───────▼────────┐ ┌──────▼───────────┐
│  MCP Servidor  │ │  MCP Servidor  │ │  MCP Servidor    │
│  (Base Datos)  │ │  (API Web)     │ │  (Sistema Arch.) │
│                │ │                │ │                   │
│  - Tools       │ │  - Tools       │ │  - Tools          │
│  - Resources   │ │  - Resources   │ │  - Resources      │
│  - Prompts     │ │  - Prompts     │ │  - Prompts        │
└────────────────┘ └────────────────┘ └───────────────────┘
```

### Componentes Principales de MCP

| Componente | Descripcion |
|------------|-------------|
| **Host** | La aplicacion que contiene al agente (ej: Claude Desktop, IDE) |
| **Cliente MCP** | Componente que mantiene conexion 1:1 con un servidor MCP |
| **Servidor MCP** | Servicio que expone herramientas, recursos y prompts |
| **Tools** | Funciones ejecutables que el modelo puede invocar |
| **Resources** | Datos que el servidor puede proporcionar al contexto |
| **Prompts** | Plantillas de prompts reutilizables |

### Cuando usar MCP vs Function Calling directo?

| Criterio | Function Calling | MCP |
|----------|-----------------|-----|
| Complejidad | Simple, pocas herramientas | Multiples servicios, ecosistema grande |
| Estandarizacion | Ad-hoc por aplicacion | Protocolo estandar |
| Descubrimiento | Manual | Automatico |
| Reutilizacion | Limitada | Alta (servidores compartidos) |
| Caso de uso ideal | Prototipos, apps simples | Produccion, integraciones enterprise |

### 5.1 Implementacion de un patron similar a MCP en Python

Vamos a implementar un patron inspirado en MCP para entender los conceptos. Crearemos un **servidor** que expone herramientas y un **cliente** que las descubre y utiliza.

In [22]:
class ServidorMCP:
    """
    Simulacion de un servidor MCP.
    Expone herramientas (tools), recursos (resources) y metadata.
    """

    def __init__(self, nombre: str, version: str = "1.0.0"):
        self.nombre = nombre
        self.version = version
        self._tools: dict[str, dict] = {}
        self._funciones: dict[str, Callable] = {}
        self._recursos: dict[str, Any] = {}

    def registrar_tool(self, nombre: str, descripcion: str, parametros: dict, handler: Callable) -> None:
        """Registra una herramienta en el servidor."""
        self._tools[nombre] = {
            "name": nombre,
            "description": descripcion,
            "inputSchema": parametros
        }
        self._funciones[nombre] = handler

    def registrar_recurso(self, uri: str, datos: Any) -> None:
        """Registra un recurso accesible por URI."""
        self._recursos[uri] = datos

    # --- Endpoints del protocolo MCP ---

    def initialize(self) -> dict:
        """Endpoint: inicializacion del servidor."""
        return {
            "protocolVersion": "2024-11-05",
            "serverInfo": {
                "name": self.nombre,
                "version": self.version
            },
            "capabilities": {
                "tools": {"listChanged": True},
                "resources": {"subscribe": False}
            }
        }

    def tools_list(self) -> dict:
        """Endpoint: listar herramientas disponibles."""
        return {"tools": list(self._tools.values())}

    def tools_call(self, nombre: str, argumentos: dict) -> dict:
        """Endpoint: ejecutar una herramienta."""
        if nombre not in self._funciones:
            return {
                "content": [{"type": "text", "text": f"Error: herramienta '{nombre}' no encontrada"}],
                "isError": True
            }
        try:
            resultado = self._funciones[nombre](**argumentos)
            return {
                "content": [{"type": "text", "text": resultado}],
                "isError": False
            }
        except Exception as e:
            return {
                "content": [{"type": "text", "text": f"Error ejecutando herramienta: {str(e)}"}],
                "isError": True
            }

    def resources_list(self) -> dict:
        """Endpoint: listar recursos disponibles."""
        return {"resources": [{"uri": uri, "name": uri} for uri in self._recursos]}

    def resources_read(self, uri: str) -> dict:
        """Endpoint: leer un recurso."""
        if uri in self._recursos:
            return {"contents": [{"uri": uri, "text": json.dumps(self._recursos[uri])}]}
        return {"error": f"Recurso '{uri}' no encontrado"}


print("Clase ServidorMCP definida.")

Clase ServidorMCP definida.


In [23]:
class ClienteMCP:
    """
    Simulacion de un cliente MCP.
    Se conecta a un servidor, descubre herramientas y las pone a disposicion del agente.
    """

    def __init__(self):
        self._servidores: dict[str, ServidorMCP] = {}
        self._tool_map: dict[str, str] = {}  # tool_name -> server_name

    def conectar(self, servidor: ServidorMCP) -> dict:
        """Conecta con un servidor MCP y descubre sus herramientas."""
        # Paso 1: Inicializar conexion
        info = servidor.initialize()
        nombre_servidor = info["serverInfo"]["name"]
        self._servidores[nombre_servidor] = servidor

        # Paso 2: Descubrir herramientas
        tools_info = servidor.tools_list()
        for tool in tools_info["tools"]:
            # Namespace: servidor.herramienta para evitar conflictos
            tool_id = f"{nombre_servidor}__{tool['name']}"
            self._tool_map[tool_id] = nombre_servidor

        return {
            "servidor": nombre_servidor,
            "version": info["serverInfo"]["version"],
            "herramientas_descubiertas": len(tools_info["tools"])
        }

    def obtener_herramientas_openai(self) -> list[dict]:
        """Convierte las herramientas MCP al formato OpenAI tools."""
        tools = []
        for nombre_servidor, servidor in self._servidores.items():
            for tool in servidor.tools_list()["tools"]:
                tool_id = f"{nombre_servidor}__{tool['name']}"
                tools.append({
                    "type": "function",
                    "function": {
                        "name": tool_id,
                        "description": f"[{nombre_servidor}] {tool['description']}",
                        "parameters": tool["inputSchema"]
                    }
                })
        return tools

    def ejecutar_herramienta(self, tool_id: str, argumentos: dict) -> str:
        """Ejecuta una herramienta a traves del servidor MCP correspondiente."""
        if tool_id not in self._tool_map:
            return json.dumps({"error": f"Herramienta '{tool_id}' no encontrada"})

        nombre_servidor = self._tool_map[tool_id]
        servidor = self._servidores[nombre_servidor]
        # Extraer nombre original de la herramienta
        nombre_tool = tool_id.replace(f"{nombre_servidor}__", "")

        resultado = servidor.tools_call(nombre_tool, argumentos)
        if resultado["isError"]:
            return json.dumps({"error": resultado["content"][0]["text"]})
        return resultado["content"][0]["text"]

    def obtener_funciones(self) -> dict[str, Callable]:
        """Retorna mapeo de funciones para usar con ejecutar_agente."""
        funciones = {}
        for tool_id in self._tool_map:
            # Crear closure para capturar tool_id
            def crear_handler(tid):
                def handler(**kwargs):
                    return self.ejecutar_herramienta(tid, kwargs)
                return handler
            funciones[tool_id] = crear_handler(tool_id)
        return funciones

    def listar_servidores(self) -> list[str]:
        """Lista servidores conectados."""
        return list(self._servidores.keys())


print("Clase ClienteMCP definida.")

Clase ClienteMCP definida.


In [24]:
# Crear servidores MCP

# Servidor 1: Matematicas
servidor_math = ServidorMCP("matematicas", "1.0.0")
servidor_math.registrar_tool(
    "calcular",
    "Realiza operaciones matematicas basicas",
    {
        "type": "object",
        "properties": {
            "operacion": {"type": "string", "enum": ["suma", "resta", "multiplicacion", "division", "potencia"]},
            "a": {"type": "number"},
            "b": {"type": "number"}
        },
        "required": ["operacion", "a", "b"]
    },
    ejecutar_calculadora
)

# Servidor 2: Informacion
servidor_info = ServidorMCP("informacion", "1.0.0")
servidor_info.registrar_tool(
    "consultar_clima",
    "Consulta el clima actual de una ciudad",
    {
        "type": "object",
        "properties": {"ciudad": {"type": "string"}},
        "required": ["ciudad"]
    },
    ejecutar_clima
)
servidor_info.registrar_tool(
    "convertir_moneda",
    "Convierte entre monedas CLP, USD y EUR",
    {
        "type": "object",
        "properties": {
            "monto": {"type": "number"},
            "de_moneda": {"type": "string", "enum": ["CLP", "USD", "EUR"]},
            "a_moneda": {"type": "string", "enum": ["CLP", "USD", "EUR"]}
        },
        "required": ["monto", "de_moneda", "a_moneda"]
    },
    ejecutar_conversor_moneda
)
# Agregar un recurso al servidor de informacion
servidor_info.registrar_recurso("config://ciudades_favoritas", ["Santiago", "Valparaiso", "Concepcion"])

# Crear cliente y conectar servidores
cliente_mcp = ClienteMCP()

print("Conectando servidores MCP...\n")
resultado1 = cliente_mcp.conectar(servidor_math)
print(f"  Conectado: {resultado1}")
resultado2 = cliente_mcp.conectar(servidor_info)
print(f"  Conectado: {resultado2}")

print(f"\nServidores conectados: {cliente_mcp.listar_servidores()}")
print(f"\nHerramientas disponibles (formato OpenAI):")
for tool in cliente_mcp.obtener_herramientas_openai():
    print(f"  - {tool['function']['name']}: {tool['function']['description']}")

Conectando servidores MCP...

  Conectado: {'servidor': 'matematicas', 'version': '1.0.0', 'herramientas_descubiertas': 1}
  Conectado: {'servidor': 'informacion', 'version': '1.0.0', 'herramientas_descubiertas': 2}

Servidores conectados: ['matematicas', 'informacion']

Herramientas disponibles (formato OpenAI):
  - matematicas__calcular: [matematicas] Realiza operaciones matematicas basicas
  - informacion__consultar_clima: [informacion] Consulta el clima actual de una ciudad
  - informacion__convertir_moneda: [informacion] Convierte entre monedas CLP, USD y EUR


In [25]:
# Usar el agente con herramientas MCP
ejecutar_agente(
    "Calcula cuanto es 250 por 12 y luego convierte el resultado de CLP a USD",
    cliente_mcp.obtener_herramientas_openai(),
    cliente_mcp.obtener_funciones()
)


Usuario: Calcula cuanto es 250 por 12 y luego convierte el resultado de CLP a USD

[Iteracion 1] Herramienta invocada: matematicas__calcular
  Argumentos: {'operacion': 'multiplicacion', 'a': 250, 'b': 12}
  Resultado: {"operacion": "multiplicacion", "a": 250, "b": 12, "resultado": 3000}

[Iteracion 2] Herramienta invocada: informacion__convertir_moneda
  Argumentos: {'monto': 3000, 'de_moneda': 'CLP', 'a_moneda': 'USD'}
  Resultado: {"monto_original": 3000, "de": "CLP", "a": "USD", "resultado": 3.3, "tasa": 0.0011}

Respuesta final del agente:
El resultado de \(250 \times 12\) es 3000. 

Al convertir 3000 CLP a USD, obtienes aproximadamente 3.3 USD.


'El resultado de \\(250 \\times 12\\) es 3000. \n\nAl convertir 3000 CLP a USD, obtienes aproximadamente 3.3 USD.'

### 5.2 Reflexion: Ventajas del patron MCP

Al usar este patron similar a MCP, observamos:

1. **Descubrimiento automatico**: El cliente descubre herramientas al conectarse, sin configuracion manual
2. **Namespace**: Las herramientas usan `servidor__herramienta` para evitar conflictos de nombres
3. **Desacoplamiento**: Los servidores son independientes, se pueden agregar o quitar sin afectar otros
4. **Estandarizacion**: Todos los servidores usan la misma interfaz

En produccion, MCP usa transporte real (stdio, HTTP/SSE) y soporta caracteristicas avanzadas como suscripciones a recursos y notificaciones.

---

## Seccion 6: Manejo de Errores en Herramientas

En produccion, las herramientas pueden fallar por multiples razones:
- APIs externas no disponibles
- Timeouts de red
- Parametros invalidos
- Limites de tasa excedidos

Es fundamental implementar estrategias robustas de manejo de errores.

In [26]:
import time
import traceback


class HerramientaRobusta:
    """
    Envoltorio para herramientas con manejo de errores robusto.
    Implementa reintentos, timeouts y degradacion elegante.
    """

    def __init__(self, nombre: str, funcion: Callable, max_reintentos: int = 3,
                 timeout_segundos: float = 10.0, fallback: Callable | None = None):
        self.nombre = nombre
        self.funcion = funcion
        self.max_reintentos = max_reintentos
        self.timeout_segundos = timeout_segundos
        self.fallback = fallback
        self.historial_errores: list[dict] = []

    def ejecutar(self, **kwargs) -> str:
        """
        Ejecuta la herramienta con reintentos y manejo de errores.
        """
        ultimo_error = None

        for intento in range(1, self.max_reintentos + 1):
            try:
                inicio = time.time()
                resultado = self.funcion(**kwargs)
                duracion = time.time() - inicio

                # Verificar timeout
                if duracion > self.timeout_segundos:
                    raise TimeoutError(f"La herramienta tardo {duracion:.2f}s (limite: {self.timeout_segundos}s)")

                print(f"    [OK] {self.nombre} ejecutada en {duracion:.3f}s (intento {intento})")
                return resultado

            except Exception as e:
                ultimo_error = e
                error_info = {
                    "intento": intento,
                    "error": str(e),
                    "tipo": type(e).__name__,
                    "timestamp": datetime.now().isoformat()
                }
                self.historial_errores.append(error_info)
                print(f"    [ERROR] {self.nombre} - Intento {intento}/{self.max_reintentos}: {e}")

                if intento < self.max_reintentos:
                    # Espera exponencial entre reintentos
                    espera = 0.5 * (2 ** (intento - 1))
                    print(f"    [RETRY] Esperando {espera}s antes de reintentar...")
                    time.sleep(espera)

        # Si todos los reintentos fallan, intentar fallback
        if self.fallback:
            print(f"    [FALLBACK] Usando funcion alternativa para {self.nombre}")
            try:
                return self.fallback(**kwargs)
            except Exception as e:
                print(f"    [FALLBACK ERROR] {e}")

        # Degradacion elegante: retornar error informativo
        return json.dumps({
            "error": f"Herramienta '{self.nombre}' no disponible despues de {self.max_reintentos} intentos",
            "ultimo_error": str(ultimo_error),
            "sugerencia": "La herramienta no esta disponible temporalmente. Intenta de nuevo mas tarde."
        })


print("Clase HerramientaRobusta definida.")

Clase HerramientaRobusta definida.


In [27]:
# Demostrar manejo de errores

# Herramienta que falla aleatoriamente
contador_llamadas = {"n": 0}

def api_inestable(consulta: str) -> str:
    """Simula una API que falla las primeras 2 veces."""
    contador_llamadas["n"] += 1
    if contador_llamadas["n"] <= 2:
        raise ConnectionError(f"Error de conexion (intento {contador_llamadas['n']})")
    return json.dumps({"resultado": f"Datos para: {consulta}", "fuente": "API estable"})

def fallback_local(consulta: str) -> str:
    """Funcion de respaldo que usa datos locales."""
    return json.dumps({"resultado": f"Datos locales (cache) para: {consulta}", "fuente": "cache local"})

# Crear herramienta robusta
herramienta_robusta = HerramientaRobusta(
    nombre="api_datos",
    funcion=api_inestable,
    max_reintentos=3,
    timeout_segundos=5.0,
    fallback=fallback_local
)

print("Ejecutando herramienta que falla las primeras 2 veces...\n")
resultado = herramienta_robusta.ejecutar(consulta="ventas Q1 2026")
print(f"\nResultado final: {resultado}")
print(f"\nHistorial de errores: {json.dumps(herramienta_robusta.historial_errores, indent=2)}")

Ejecutando herramienta que falla las primeras 2 veces...

    [ERROR] api_datos - Intento 1/3: Error de conexion (intento 1)
    [RETRY] Esperando 0.5s antes de reintentar...
    [ERROR] api_datos - Intento 2/3: Error de conexion (intento 2)
    [RETRY] Esperando 1.0s antes de reintentar...
    [OK] api_datos ejecutada en 0.000s (intento 3)

Resultado final: {"resultado": "Datos para: ventas Q1 2026", "fuente": "API estable"}

Historial de errores: [
  {
    "intento": 1,
    "error": "Error de conexion (intento 1)",
    "tipo": "ConnectionError",
    "timestamp": "2026-04-28T12:45:28.619614"
  },
  {
    "intento": 2,
    "error": "Error de conexion (intento 2)",
    "tipo": "ConnectionError",
    "timestamp": "2026-04-28T12:45:29.124699"
  }
]


In [28]:
# Demostrar fallback cuando todos los reintentos fallan

def api_caida(consulta: str) -> str:
    """Simula una API que siempre falla."""
    raise ConnectionError("Servicio no disponible - Error 503")

herramienta_con_fallback = HerramientaRobusta(
    nombre="api_caida",
    funcion=api_caida,
    max_reintentos=2,  # Solo 2 reintentos para no esperar mucho
    fallback=fallback_local
)

print("Ejecutando herramienta que siempre falla (con fallback)...\n")
resultado = herramienta_con_fallback.ejecutar(consulta="reporte mensual")
print(f"\nResultado (via fallback): {resultado}")

Ejecutando herramienta que siempre falla (con fallback)...

    [ERROR] api_caida - Intento 1/2: Servicio no disponible - Error 503
    [RETRY] Esperando 0.5s antes de reintentar...
    [ERROR] api_caida - Intento 2/2: Servicio no disponible - Error 503
    [FALLBACK] Usando funcion alternativa para api_caida

Resultado (via fallback): {"resultado": "Datos locales (cache) para: reporte mensual", "fuente": "cache local"}


In [29]:
# Demostrar degradacion elegante sin fallback

herramienta_sin_fallback = HerramientaRobusta(
    nombre="api_critica",
    funcion=api_caida,
    max_reintentos=2,
    fallback=None  # Sin fallback
)

print("Ejecutando herramienta que siempre falla (sin fallback)...\n")
resultado = herramienta_sin_fallback.ejecutar(consulta="datos criticos")
print(f"\nResultado (degradacion elegante): {resultado}")

Ejecutando herramienta que siempre falla (sin fallback)...

    [ERROR] api_critica - Intento 1/2: Servicio no disponible - Error 503
    [RETRY] Esperando 0.5s antes de reintentar...
    [ERROR] api_critica - Intento 2/2: Servicio no disponible - Error 503

Resultado (degradacion elegante): {"error": "Herramienta 'api_critica' no disponible despues de 2 intentos", "ultimo_error": "Servicio no disponible - Error 503", "sugerencia": "La herramienta no esta disponible temporalmente. Intenta de nuevo mas tarde."}


In [30]:
# Integrar herramientas robustas con el agente

# Crear herramientas robustas para el agente
calc_robusta = HerramientaRobusta("calculadora", ejecutar_calculadora)
clima_robusto = HerramientaRobusta("clima", ejecutar_clima)

# Wrapper que usa la herramienta robusta
def calc_robusta_wrapper(operacion: str, a: float, b: float) -> str:
    return calc_robusta.ejecutar(operacion=operacion, a=a, b=b)

def clima_robusto_wrapper(ciudad: str) -> str:
    return clima_robusto.ejecutar(ciudad=ciudad)

funciones_robustas = {
    "calculadora": calc_robusta_wrapper,
    "clima": clima_robusto_wrapper
}

herramientas_basicas = [
    {
        "type": "function",
        "function": {
            "name": "calculadora",
            "description": "Realiza operaciones matematicas",
            "parameters": {
                "type": "object",
                "properties": {
                    "operacion": {"type": "string", "enum": ["suma", "resta", "multiplicacion", "division", "potencia"]},
                    "a": {"type": "number"},
                    "b": {"type": "number"}
                },
                "required": ["operacion", "a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "clima",
            "description": "Consulta el clima de una ciudad",
            "parameters": {
                "type": "object",
                "properties": {"ciudad": {"type": "string"}},
                "required": ["ciudad"]
            }
        }
    }
]

print("Ejecutando agente con herramientas robustas...")
ejecutar_agente(
    "Cuanto es 1500 dividido en 7? Y como esta el clima en Temuco?",
    herramientas_basicas,
    funciones_robustas
)

Ejecutando agente con herramientas robustas...

Usuario: Cuanto es 1500 dividido en 7? Y como esta el clima en Temuco?

[Iteracion 1] Herramienta invocada: calculadora
  Argumentos: {'operacion': 'division', 'a': 1500, 'b': 7}
    [OK] calculadora ejecutada en 0.000s (intento 1)
  Resultado: {"operacion": "division", "a": 1500, "b": 7, "resultado": 214.28571428571428}

[Iteracion 1] Herramienta invocada: clima
  Argumentos: {'ciudad': 'Temuco'}
    [OK] clima ejecutada en 0.000s (intento 1)
  Resultado: {"ciudad": "Temuco", "temperatura_celsius": 23.1, "condicion": "Soleado", "humedad_porcentaje": 44, "viento_kmh": 26.8}

Respuesta final del agente:
La división de 1500 entre 7 es aproximadamente **214.29**.

En cuanto al clima en Temuco, actualmente está **soleado** con una temperatura de **23.1°C**, humedad del **44%** y vientos a **26.8 km/h**.


'La división de 1500 entre 7 es aproximadamente **214.29**.\n\nEn cuanto al clima en Temuco, actualmente está **soleado** con una temperatura de **23.1°C**, humedad del **44%** y vientos a **26.8 km/h**.'

---

## Ejercicio Final: Asistente de Productividad

### Instrucciones

Implementa un **Asistente de Productividad** completo que integre las siguientes herramientas:

1. **Calculadora**: Para operaciones matematicas
2. **Notas con persistencia**: Guardar y buscar notas que persisten entre interacciones
3. **Gestor de tareas**: Crear, listar, completar y eliminar tareas con prioridades

El asistente debe:
- Manejar tareas multi-paso (ej: "calcula el total, guardalo como nota y crea una tarea de seguimiento")
- Mantener estado entre interacciones
- Manejar errores de forma elegante

### Requisitos
- Usa el patron `RegistroHerramientas` para registrar las herramientas
- Implementa al menos 6 herramientas (2 por categoria)
- Demuestra al menos 3 interacciones que usen multiples herramientas

In [31]:
# --- EJERCICIO FINAL: Asistente de Productividad ---

# Almacenes de estado
almacen_notas = []
almacen_tareas = []
id_tarea_counter = {"valor": 0}


# === HERRAMIENTAS DE CALCULADORA ===

def prod_calcular(operacion: str, a: float, b: float) -> str:
    """Calculadora basica."""
    ops = {
        "suma": lambda x, y: x + y,
        "resta": lambda x, y: x - y,
        "multiplicacion": lambda x, y: x * y,
        "division": lambda x, y: x / y if y != 0 else "Error: division por cero",
        "potencia": lambda x, y: x ** y,
        "porcentaje": lambda x, y: x * y / 100
    }
    if operacion not in ops:
        return json.dumps({"error": f"Operacion '{operacion}' no soportada"})
    resultado = ops[operacion](a, b)
    return json.dumps({"operacion": operacion, "a": a, "b": b, "resultado": resultado})


def prod_calcular_promedio(valores: list[float]) -> str:
    """Calcula el promedio de una lista de numeros."""
    if not valores:
        return json.dumps({"error": "La lista de valores esta vacia"})
    promedio = sum(valores) / len(valores)
    return json.dumps({"valores": valores, "promedio": round(promedio, 2), "cantidad": len(valores)})


# === HERRAMIENTAS DE NOTAS ===

def prod_guardar_nota(titulo: str, contenido: str, categoria: str = "general") -> str:
    """Guarda una nota con titulo, contenido y categoria."""
    nota = {
        "id": len(almacen_notas) + 1,
        "titulo": titulo,
        "contenido": contenido,
        "categoria": categoria,
        "fecha": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    almacen_notas.append(nota)
    return json.dumps({"mensaje": f"Nota '{titulo}' guardada", "id": nota["id"], "categoria": categoria})


def prod_buscar_notas(termino: str = "", categoria: str = "") -> str:
    """Busca notas por termino en titulo/contenido o por categoria. Sin parametros lista todas."""
    resultados = almacen_notas
    if termino:
        resultados = [n for n in resultados
                      if termino.lower() in n["titulo"].lower() or termino.lower() in n["contenido"].lower()]
    if categoria:
        resultados = [n for n in resultados if n["categoria"].lower() == categoria.lower()]
    return json.dumps({"notas": resultados, "total": len(resultados)})


# === HERRAMIENTAS DE TAREAS ===

def prod_crear_tarea(descripcion: str, prioridad: str = "media", categoria: str = "general") -> str:
    """Crea una nueva tarea con descripcion, prioridad y categoria."""
    id_tarea_counter["valor"] += 1
    tarea = {
        "id": id_tarea_counter["valor"],
        "descripcion": descripcion,
        "prioridad": prioridad,
        "categoria": categoria,
        "completada": False,
        "fecha_creacion": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    almacen_tareas.append(tarea)
    return json.dumps({"mensaje": "Tarea creada", "id": tarea["id"], "descripcion": descripcion})


def prod_gestionar_tarea(id_tarea: int, accion: str) -> str:
    """Gestiona una tarea: completar o eliminar."""
    for i, tarea in enumerate(almacen_tareas):
        if tarea["id"] == id_tarea:
            if accion == "completar":
                tarea["completada"] = True
                tarea["fecha_completada"] = datetime.now().strftime("%Y-%m-%d %H:%M")
                return json.dumps({"mensaje": f"Tarea {id_tarea} completada", "descripcion": tarea["descripcion"]})
            elif accion == "eliminar":
                eliminada = almacen_tareas.pop(i)
                return json.dumps({"mensaje": f"Tarea {id_tarea} eliminada", "descripcion": eliminada["descripcion"]})
            else:
                return json.dumps({"error": f"Accion '{accion}' no valida. Usa 'completar' o 'eliminar'"})
    return json.dumps({"error": f"Tarea con id {id_tarea} no encontrada"})


def prod_listar_tareas(solo_pendientes: bool = False, categoria: str = "") -> str:
    """Lista tareas. Puede filtrar por pendientes y por categoria."""
    resultado = almacen_tareas
    if solo_pendientes:
        resultado = [t for t in resultado if not t["completada"]]
    if categoria:
        resultado = [t for t in resultado if t["categoria"].lower() == categoria.lower()]
    return json.dumps({
        "tareas": resultado,
        "total": len(almacen_tareas),
        "pendientes": sum(1 for t in almacen_tareas if not t["completada"]),
        "completadas": sum(1 for t in almacen_tareas if t["completada"])
    })


print("Funciones del asistente de productividad definidas.")

Funciones del asistente de productividad definidas.


In [32]:
# Registrar todas las herramientas
asistente = RegistroHerramientas()

print("Registrando herramientas del asistente de productividad...\n")

# Calculadora
asistente.registrar(
    "calcular", "Realiza operaciones matematicas: suma, resta, multiplicacion, division, potencia, porcentaje.",
    {
        "type": "object",
        "properties": {
            "operacion": {"type": "string", "enum": ["suma", "resta", "multiplicacion", "division", "potencia", "porcentaje"], "description": "Operacion a realizar. Para porcentaje: calcula a% de b, es decir (a*b)/100."},
            "a": {"type": "number", "description": "Primer operando"},
            "b": {"type": "number", "description": "Segundo operando"}
        },
        "required": ["operacion", "a", "b"]
    },
    prod_calcular
)

asistente.registrar(
    "calcular_promedio", "Calcula el promedio de una lista de numeros.",
    {
        "type": "object",
        "properties": {
            "valores": {"type": "array", "items": {"type": "number"}, "description": "Lista de numeros para calcular el promedio"}
        },
        "required": ["valores"]
    },
    prod_calcular_promedio
)

# Notas
asistente.registrar(
    "guardar_nota", "Guarda una nota con titulo, contenido y categoria opcional.",
    {
        "type": "object",
        "properties": {
            "titulo": {"type": "string", "description": "Titulo de la nota"},
            "contenido": {"type": "string", "description": "Contenido de la nota"},
            "categoria": {"type": "string", "description": "Categoria: trabajo, personal, estudio, general"}
        },
        "required": ["titulo", "contenido"]
    },
    prod_guardar_nota
)

asistente.registrar(
    "buscar_notas", "Busca notas por termino en titulo/contenido o por categoria. Sin parametros lista todas.",
    {
        "type": "object",
        "properties": {
            "termino": {"type": "string", "description": "Texto a buscar en titulo o contenido"},
            "categoria": {"type": "string", "description": "Filtrar por categoria"}
        }
    },
    prod_buscar_notas
)

# Tareas
asistente.registrar(
    "crear_tarea", "Crea una nueva tarea con descripcion, prioridad (alta/media/baja) y categoria.",
    {
        "type": "object",
        "properties": {
            "descripcion": {"type": "string", "description": "Descripcion de la tarea"},
            "prioridad": {"type": "string", "enum": ["alta", "media", "baja"], "description": "Nivel de prioridad"},
            "categoria": {"type": "string", "description": "Categoria de la tarea"}
        },
        "required": ["descripcion"]
    },
    prod_crear_tarea
)

asistente.registrar(
    "gestionar_tarea", "Gestiona una tarea existente: completar o eliminar, usando su ID.",
    {
        "type": "object",
        "properties": {
            "id_tarea": {"type": "integer", "description": "ID de la tarea"},
            "accion": {"type": "string", "enum": ["completar", "eliminar"], "description": "Accion a realizar"}
        },
        "required": ["id_tarea", "accion"]
    },
    prod_gestionar_tarea
)

asistente.registrar(
    "listar_tareas", "Lista tareas. Puede filtrar solo pendientes y por categoria.",
    {
        "type": "object",
        "properties": {
            "solo_pendientes": {"type": "boolean", "description": "Mostrar solo tareas pendientes"},
            "categoria": {"type": "string", "description": "Filtrar por categoria"}
        }
    },
    prod_listar_tareas
)

print(f"\nAsistente listo: {asistente}")

Registrando herramientas del asistente de productividad...

  [+] Herramienta 'calcular' registrada exitosamente
  [+] Herramienta 'calcular_promedio' registrada exitosamente
  [+] Herramienta 'guardar_nota' registrada exitosamente
  [+] Herramienta 'buscar_notas' registrada exitosamente
  [+] Herramienta 'crear_tarea' registrada exitosamente
  [+] Herramienta 'gestionar_tarea' registrada exitosamente
  [+] Herramienta 'listar_tareas' registrada exitosamente

Asistente listo: RegistroHerramientas(7 herramientas: ['calcular', 'calcular_promedio', 'guardar_nota', 'buscar_notas', 'crear_tarea', 'gestionar_tarea', 'listar_tareas'])


In [33]:
# Interaccion 1: Tarea multi-paso con calculo y notas
ejecutar_agente(
    (
        "Tengo 3 proyectos con costos de $150.000, $230.000 y $185.000 CLP. "
        "Calcula el promedio de costos y guarda una nota con el titulo 'Presupuesto Proyectos' "
        "en la categoria 'trabajo' con el resultado del calculo."
    ),
    asistente.obtener_definiciones(),
    asistente.obtener_funciones()
)


Usuario: Tengo 3 proyectos con costos de $150.000, $230.000 y $185.000 CLP. Calcula el promedio de costos y guarda una nota con el titulo 'Presupuesto Proyectos' en la categoria 'trabajo' con el resultado del calculo.

[Iteracion 1] Herramienta invocada: calcular_promedio
  Argumentos: {'valores': [150000, 230000, 185000]}
  Resultado: {"valores": [150000, 230000, 185000], "promedio": 188333.33, "cantidad": 3}

[Iteracion 2] Herramienta invocada: guardar_nota
  Argumentos: {'titulo': 'Presupuesto Proyectos', 'contenido': 'El promedio de costos de los proyectos es de $188,333.33 CLP.', 'categoria': 'trabajo'}
  Resultado: {"mensaje": "Nota 'Presupuesto Proyectos' guardada", "id": 1, "categoria": "trabajo"}

Respuesta final del agente:
El promedio de costos de los proyectos es de $188,333.33 CLP y he guardado una nota titulada "Presupuesto Proyectos" en la categoría "trabajo" con este resultado.


'El promedio de costos de los proyectos es de $188,333.33 CLP y he guardado una nota titulada "Presupuesto Proyectos" en la categoría "trabajo" con este resultado.'

In [34]:
# Interaccion 2: Crear tareas de seguimiento
ejecutar_agente(
    (
        "Basandote en los proyectos, crea las siguientes tareas de trabajo: "
        "1) 'Revisar presupuesto proyecto 1' con prioridad alta, "
        "2) 'Contactar proveedor proyecto 2' con prioridad media, "
        "3) 'Preparar informe consolidado' con prioridad alta. "
        "Todas en categoria 'trabajo'. Luego muestrame la lista completa."
    ),
    asistente.obtener_definiciones(),
    asistente.obtener_funciones()
)


Usuario: Basandote en los proyectos, crea las siguientes tareas de trabajo: 1) 'Revisar presupuesto proyecto 1' con prioridad alta, 2) 'Contactar proveedor proyecto 2' con prioridad media, 3) 'Preparar informe consolidado' con prioridad alta. Todas en categoria 'trabajo'. Luego muestrame la lista completa.

[Iteracion 1] Herramienta invocada: crear_tarea
  Argumentos: {'descripcion': 'Revisar presupuesto proyecto 1', 'prioridad': 'alta', 'categoria': 'trabajo'}
  Resultado: {"mensaje": "Tarea creada", "id": 1, "descripcion": "Revisar presupuesto proyecto 1"}

[Iteracion 1] Herramienta invocada: crear_tarea
  Argumentos: {'descripcion': 'Contactar proveedor proyecto 2', 'prioridad': 'media', 'categoria': 'trabajo'}
  Resultado: {"mensaje": "Tarea creada", "id": 2, "descripcion": "Contactar proveedor proyecto 2"}

[Iteracion 1] Herramienta invocada: crear_tarea
  Argumentos: {'descripcion': 'Preparar informe consolidado', 'prioridad': 'alta', 'categoria': 'trabajo'}
  Resultado: {"mensa

'He creado las tareas en la categoría "trabajo" como solicitaste. Aquí tienes la lista completa de las tareas:\n\n1. **Revisar presupuesto proyecto 1**\n   - Prioridad: Alta\n   - Completada: No\n\n2. **Contactar proveedor proyecto 2**\n   - Prioridad: Media\n   - Completada: No\n\n3. **Preparar informe consolidado**\n   - Prioridad: Alta\n   - Completada: No\n\nTotal de tareas: 3  \nTareas pendientes: 3  \nTareas completadas: 0'

In [35]:
# Interaccion 3: Gestionar tareas y documentar
ejecutar_agente(
    (
        "Marca la tarea 1 como completada. Luego calcula que porcentaje de las tareas "
        "totales esta completado (si hay 3 tareas y 1 completada, es 1/3). "
        "Guarda una nota con titulo 'Progreso Semanal' en categoria 'trabajo' "
        "con el porcentaje de avance. Finalmente muestrame las tareas pendientes."
    ),
    asistente.obtener_definiciones(),
    asistente.obtener_funciones()
)


Usuario: Marca la tarea 1 como completada. Luego calcula que porcentaje de las tareas totales esta completado (si hay 3 tareas y 1 completada, es 1/3). Guarda una nota con titulo 'Progreso Semanal' en categoria 'trabajo' con el porcentaje de avance. Finalmente muestrame las tareas pendientes.

[Iteracion 1] Herramienta invocada: gestionar_tarea
  Argumentos: {'id_tarea': 1, 'accion': 'completar'}
  Resultado: {"mensaje": "Tarea 1 completada", "descripcion": "Revisar presupuesto proyecto 1"}

[Iteracion 2] Herramienta invocada: listar_tareas
  Argumentos: {'solo_pendientes': True}
  Resultado: {"tareas": [{"id": 2, "descripcion": "Contactar proveedor proyecto 2", "prioridad": "media", "categoria": "trabajo", "completada": false, "fecha_creacion": "2026-04-28 12:45"}, {"id": 3, "descripcion": "Preparar informe consolidado", "prioridad": "alta", "categoria": "trabajo", "completada": false, "fecha_creacion": "2026-04-28 12:45"}], "total": 3, "pendientes": 2, "completadas": 1}

[Iteracion 

"He completado la tarea 1. De las 3 tareas totales, 1 ha sido completada, lo que representa un avance del **33.33%**. Esta información se ha guardado en una nota titulada **'Progreso Semanal'** en la categoría **trabajo**.\n\nActualmente, estas son las tareas pendientes:\n\n1. **Contactar proveedor proyecto 2** (Prioridad: Media)\n2. **Preparar informe consolidado** (Prioridad: Alta) \n\nSi necesitas algo más, no dudes en decírmelo."

In [36]:
# Interaccion 4: Consultar notas guardadas
ejecutar_agente(
    "Busca todas las notas de la categoria 'trabajo' y muestrame un resumen.",
    asistente.obtener_definiciones(),
    asistente.obtener_funciones()
)


Usuario: Busca todas las notas de la categoria 'trabajo' y muestrame un resumen.

[Iteracion 1] Herramienta invocada: buscar_notas
  Argumentos: {'categoria': 'trabajo'}
  Resultado: {"notas": [{"id": 1, "titulo": "Presupuesto Proyectos", "contenido": "El promedio de costos de los proyectos es de $188,333.33 CLP.", "categoria": "trabajo", "fecha": "2026-04-28 12:45"}, {"id": 2, "titulo": "Progreso Semanal", "contenido": "El porcentaje de avance es del 33.33%.", "categoria": "trabajo", "fecha": "2026-04-28 12:46"}], "total": 2}

Respuesta final del agente:
He encontrado 2 notas en la categoría 'trabajo'. Aquí tienes un resumen:

1. **Título:** Presupuesto Proyectos  
   **Contenido:** El promedio de costos de los proyectos es de $188,333.33 CLP.  
   **Fecha:** 28 de abril de 2026

2. **Título:** Progreso Semanal  
   **Contenido:** El porcentaje de avance es del 33.33%.  
   **Fecha:** 28 de abril de 2026

Si necesitas más información o acciones sobre estas notas, házmelo saber.


"He encontrado 2 notas en la categoría 'trabajo'. Aquí tienes un resumen:\n\n1. **Título:** Presupuesto Proyectos  \n   **Contenido:** El promedio de costos de los proyectos es de $188,333.33 CLP.  \n   **Fecha:** 28 de abril de 2026\n\n2. **Título:** Progreso Semanal  \n   **Contenido:** El porcentaje de avance es del 33.33%.  \n   **Fecha:** 28 de abril de 2026\n\nSi necesitas más información o acciones sobre estas notas, házmelo saber."

---

## Resumen

En este notebook aprendimos:

1. **Function Calling**: Como definir herramientas con JSON Schema y ejecutarlas en un loop agente-herramienta
2. **Registro Dinamico**: Como crear un sistema flexible que permite agregar y remover herramientas en tiempo de ejecucion
3. **Herramientas con Estado**: Como mantener estado persistente entre interacciones del agente
4. **Orquestacion**: Como el agente puede encadenar multiples herramientas para resolver tareas complejas
5. **MCP (Model Context Protocol)**: La arquitectura de servidores y clientes para integracion estandarizada de herramientas
6. **Manejo de Errores**: Estrategias de reintentos, fallbacks y degradacion elegante

### Conceptos Clave para Recordar

- Las herramientas **extienden las capacidades** del agente mas alla de la generacion de texto
- El **JSON Schema** es fundamental para que el modelo entienda como usar cada herramienta
- El **Tool Execution Loop** es el patron central: solicitud -> decision -> ejecucion -> integracion
- **MCP** estandariza la integracion y facilita el descubrimiento automatico de herramientas
- El **manejo de errores** es critico en produccion: siempre implementa reintentos y fallbacks

### Proximos Pasos

- Explorar servidores MCP reales con bibliotecas como `mcp`
- Integrar herramientas con APIs externas reales
- Implementar autenticacion y autorizacion para herramientas
- Agregar observabilidad y logging a las herramientas